# CEDA 토론 시스템 테스트

AI 에이전트들이 CEDA 형식으로 토론하는 시스템을 테스트합니다.

- 10개 토론 턴 + 1 심판 판정 = 총 11턴
- 긍정측/부정측 State 격리
- 측별 모델 지정 가능

## 1. Import 및 환경 확인

In [1]:
import sys
sys.path.insert(0, '../..')

from Agent_Structure.debate import (
    run_debate,
    stream_debate,
    create_debate,
    DebateResult,
    DebateState,
    CEDA_ROUNDS,
)

print(f"CEDA 라운드: {len(CEDA_ROUNDS)}개")
for i, r in enumerate(CEDA_ROUNDS, 1):
    print(f"  {i}. [{r['round_id']}] {r['speaker']} - {r['speech_type']}")

CEDA 라운드: 8개
  1. [1AC] affirmative - constructive
  2. [CX_1AC_Q] negative - cx_question
  3. [CX_1AC_A] affirmative - cx_answer
  4. [1NC] negative - constructive
  5. [CX_1NC_Q] affirmative - cx_question
  6. [CX_1NC_A] negative - cx_answer
  7. [1AR] affirmative - rebuttal
  8. [1NR] negative - rebuttal


In [2]:
# 환경변수 확인
from Agent_Structure.config import settings

print(f"Provider: {settings.default_provider}")
print(f"Model: {settings.default_model}")
print(f"API Key 설정됨: {bool(settings.anthropic_api_key)}")

Provider: anthropic
Model: claude-sonnet-4-5-20250929
API Key 설정됨: True


## 2. 스트리밍 토론 실행

라운드별로 결과를 확인할 수 있어 진행 상황을 추적하기 좋습니다.

In [3]:
PROPOSITION = "예술 작품의 가치는 창작자의 도덕성과 분리되어야 한다"

speaker_labels = {"affirmative": "긍정측", "negative": "부정측", "judge": "심판"}
type_labels = {
    "constructive": "입론",
    "cx_question": "교차조사 질문",
    "cx_answer": "교차조사 답변",
    "rebuttal": "반박",
    "verdict": "판정",
}

speeches = []

for speech in stream_debate(PROPOSITION):
    speeches.append(speech)
    speaker = speaker_labels.get(speech['speaker'], speech['speaker'])
    stype = type_labels.get(speech['speech_type'], speech['speech_type'])
    
    print(f"\n{'='*60}")
    print(f"[{speech['round_id']}] {speaker} — {stype}")
    print(f"{'='*60}")
    print(speech['content'])

print(f"\n\n총 {len(speeches)}개 발언 완료")


[1AC] 긍정측 — 입론
# 1AC 입론

## 논점 1: 예술의 본질적 가치는 작품 자체에 존재한다

예술 작품의 가치는 작품이 담고 있는 미적 요소, 기술적 완성도, 그리고 관람자에게 전달하는 메시지에서 비롯됩니다. 카라바조의 <성 마태오의 소명>은 살인을 저지른 화가가 그렸지만, 빛과 그림자의 혁신적 사용은 바로크 미술의 기초가 되었습니다. 작품의 예술사적 기여와 미적 성취는 창작자의 범죄와 독립적으로 존재합니다.

## 논점 2: 도덕성 기준의 주관성은 예술 평가를 왜곡한다

도덕 기준은 시대와 문화에 따라 변화합니다. 피카소의 여성 편력, 바그너의 반유대주의는 현대 기준으로 비난받지만, 이를 작품 평가에 적용하면 《게르니카》와 《니벨룽의 반지》의 가치를 부정해야 합니다. 창작자의 도덕성을 기준으로 삼으면, 우리는 시대마다 예술사를 다시 써야 하는 모순에 빠집니다.

## 논점 3: 분리 원칙은 더 많은 사회적 대화를 가능하게 한다

문제적 창작자의 작품을 인정하면서도 그들의 행위를 비판할 수 있습니다. 우디 앨런의 영화를 연구하면서 그의 사생활을 비판하는 것은 양립 가능합니다. 작품을 매개로 도덕적 문제를 토론할 기회를 얻으며, 전면 거부는 오히려 대화의 장을 차단합니다.

[CX_1AC_Q] 부정측 — 교차조사 질문
# CX_1AC_Q: 긍정측 입론에 대한 교차조사

## 질문 1: 작품의 가치 평가 기준에 관하여
긍정측은 예술의 가치를 "미적 요소, 기술적 완성도, 메시지"라 했습니다. 그렇다면 기술적으로 뛰어나지만 메시지가 인종차별이나 여성혐오를 직접 담고 있다면, 메시지와 창작자의 도덕성을 어떻게 분리할 수 있습니까?

## 질문 2: 분리 가능성의 실질적 한계
카라바조와 피카소는 개인적 일탈이 작품에 직접 반영되지 않은 경우입니다. 하지만 리펜슈탈의 《의지의 승리》처럼 작품 자체가 나치 선전물이거나, 창작 과정에서 아동 학대가 발생한 영화처럼 작품과 범죄가 직접 연결된 경우에도 분리가 가능합니까?

## 질문 3: 도덕적 기준의 주관성 논

LLM 요약도 862자 초과 → 강제 절단



[CX_1NC_A] 부정측 — 교차조사 답변
# CX_1NC_A: 교차조사 답변

## 질문 1에 대한 답변
바흐 사례는 제 논점을 지지합니다. <마태 수난곡>에서 신학 구조를 모르면 코랄 반복이나 음악적 긴장의 의미를 이해할 수 없습니다. 부분적 감상은 가능하지만 "분리 가능"과 다릅니다. 더 중요한 것은 바흐는 범죄를 저지르지 않았습니다. 폴란스키는 미성년자 성폭행범이며, 그의 영화에서 소녀를 바라보는 시선은 범죄 행위와 직접 연결됩니다.

## 질문 2에 대한 답변
전제가 잘못되었습니다. 저는 실제 범죄를 저지른 창작자의 작품 내 패턴을 지적했습니다. 폴란스키는 13세 소녀 성폭행범이고, 그의 영화는 나이 차이 나는 남녀 관계를 반복 미화합니다. 이는 범죄자의 왜곡된 세계관이 작품에 투영된 증거입니다. 범죄자가 자신의 범죄를 정당화하는 서사를 반복 생산할 때는 연결을 봐야 합니다.

## 질문 3에 대한 답변
범죄 수익 환수법은 실효성이 낮습니다. 더 본질적 문제는 문화적 권력입니다. 폴란스키가 2020년 세자르상을 받은 것은 "예술과 예술가는 별개"라는 논리 때문입니다. 작품 가치를 인정하는 순간, 사회는 가해자에게 계속 발언권과 영향력을 부여합니다. 제도로는 이 구조적 권력을 차단할 수 없습니다.

## 질문 4에 대한 답변
"학술 연구용 제한적 접근"은 누가 결정합니까? 실제로는 "연구" 명목으로 계속 소비되며, 가해자는 "논쟁적 예술가"로 남습니다. 피해자 보호의 최선은 가해자를 문화적으로 주변화하는 것입니다.

## 질문 5에 대한 답변
역사적 인물과 현대 범죄자는 다릅니다.

[1AR] 긍정측 — 반박
# 1AR 최종 반박

## 핵심 쟁점 정리: 분리 가능성과 실질적 해결 가능성

부정측은 세 가지 치명적 오류를 범했습니다.

**첫째, 작품과 창작자의 본질적 혼동.**
부정측은 폴란스키 영화의 "시선"이 범죄와 연결된다 주장하나, 이는 허구와 현실을 의도적으로 혼동하는 것입니다. 교차조사에서 "실제 범죄를 저지른 창작자의 패턴"을 지적했으

## 3. 일괄 실행 (run_debate)

전체 토론을 한 번에 실행하고 `DebateResult`로 받습니다.

In [ ]:
result = run_debate("인공지능 개발에 대한 국제적 규제가 필요하다")

print(f"논제: {result.proposition}")
print(f"총 발언 수: {len(result.transcript)}")
print(f"\n{'='*60}")
print("전체 토론 기록")
print(f"{'='*60}")
print(result.format_transcript())
print(f"\n{'='*60}")
print("최종 판정")
print(f"{'='*60}")
print(result.verdict)

## 4. 측별 모델 지정 (모델 대결)

긍정측과 부정측에 서로 다른 LLM을 사용할 수 있습니다.

In [ ]:
# 예시: 같은 프로바이더 내에서 모델만 변경
# OpenAI API 키가 있는 경우 아래 주석을 해제하세요

# result = run_debate(
#     "인공지능 개발에 대한 국제적 규제가 필요하다",
#     aff_provider_name="anthropic",
#     aff_model_name="claude-sonnet-4-5-20250929",
#     neg_provider_name="openai",
#     neg_model_name="gpt-4o",
# )
# print(result.format_transcript())

## 5. 도구 포함 토론

토론 에이전트에게 웹 검색 등 도구를 제공할 수 있습니다.

In [ ]:
# 도구 포함 토론 (TAVILY_API_KEY 필요)

# from Agent_Structure.tools import tool_registry
# search_tools = tool_registry.get_by_tag("search")
# print(f"사용 가능한 검색 도구: {[t.name for t in search_tools]}")

# result = run_debate(
#     "원자력 발전소를 확대해야 한다",
#     tools=search_tools,
# )
# print(result.format_transcript())

## 6. 그래프 구조 확인

In [ ]:
graph, initial_state = create_debate("테스트 논제")

print("초기 상태:")
for k, v in initial_state.items():
    if k == 'round_sequence':
        print(f"  {k}: {len(v)}개 라운드")
    else:
        print(f"  {k}: {repr(v)[:80]}")

print(f"\n그래프 노드: {graph.get_graph().nodes}")